# Assignment 11 — Fine-Tuning GPT-2 on a Custom Text Style

**Name:** Soham Sen  
**Model:** GPT-2 (`gpt2`)  
**Custom style:** Formal Premier League disciplinary / regulatory reporting  

### Objective
Fine-tune a pretrained GPT-2 model on approximately 200–300 lines of consistent text and compare its output with the original GPT-2 model using the same prompt from Assignment 2.

### Text source
The training text is taken from the official Premier League statement published on **29 September 2026** concerning Manchester City FC and the 115 financial-rule charges. The statement is used because it is a long, consistently written primary-source document with a formal legal and regulatory style. The source also explains the findings, investigation history, sanction process and appeal process.

**Source:** https://www.premierleague.com/en/news/4727779/premier-league-statement-manchester-city-fc

> The notebook downloads and cleans the source at runtime instead of embedding the full article in the notebook. This keeps the notebook compact and lets the source text remain in its original location.


In [1]:
!pip install -q transformers datasets accelerate beautifulsoup4 requests trafilatura

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.7/254.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.8 MB/s eta 0:00:00


In [2]:
import re
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
import trafilatura

SOURCE_URL = "https://www.premierleague.com/en/news/4727779/premier-league-statement-manchester-city-fc"
DATASET_FILE = "man_city_115_style_dataset.txt"

print("Source:", SOURCE_URL)


Source: https://www.premierleague.com/en/news/4727779/premier-league-statement-manchester-city-fc


## 1. Download and extract the source text

The next cell attempts two extraction methods. It first uses `trafilatura`, then falls back to BeautifulSoup if necessary. Navigation, scripts, styles and obvious non-language content are removed.


In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/155.0 Safari/537.36"
}

response = requests.get(SOURCE_URL, headers=headers, timeout=30)
response.raise_for_status()
html = response.text

article_text = trafilatura.extract(
    html,
    include_comments=False,
    include_tables=False,
    include_links=False,
    favor_precision=True
)

if not article_text or len(article_text) < 2000:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "nav", "footer", "header"]):
        tag.decompose()

    paragraphs = []
    for p in soup.select("article p, main p, p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    unique_paragraphs = []
    seen = set()
    for p in paragraphs:
        if p not in seen:
            unique_paragraphs.append(p)
            seen.add(p)

    article_text = "\n".join(unique_paragraphs)

if not article_text:
    raise RuntimeError(
        "The article could not be extracted. Open the source URL in a browser, "
        "save/copy the article text, and place it in a file named "
        "man_city_115_source.txt in Colab, then rerun the manual fallback cell."
    )

print("Extracted characters:", len(article_text))
print("\nFirst 1500 characters:\n")
print(article_text[:1500])


Extracted characters: 9483

First 1500 characters:

An independent Commission has found Manchester City FC guilty of all charges related to serious breaches of the Premier League’s financial rules over a nine-season period, and the majority of charges in relation to its failure to co-operate with the League’s investigation.
The independent Commission found that between Seasons 2009/10 and 2017/18:
- Manchester City arranged ‘sham’ contracts (which misrepresented the true agreement between the parties) with a number of its commercial partners, as well as relying on ‘sham’ agreements with others, to artificially inflate the club’s revenues and reduce its costs
- The club filed misstated accounts and concealed the true state of its finances from its auditors and football regulators
- Manchester City was significantly in breach of both the Premier League’s and UEFA’s spending limits
- During the Premier League’s investigation, Manchester City committed multiple breaches of its duties of co

## 2. Clean the text and create approximately 200–300 training lines

The assignment requires a consistent text style and removal of unwanted formatting. The code below removes bullets, excessive whitespace, URLs, repeated spaces and obvious page elements. Long paragraphs are split into sentence-level lines. If the source produces fewer than 200 usable lines, longer sentences are split into short word-based chunks until the target range is reached.


In [8]:
def clean_text(text):
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = text.replace("•", " ")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()


# Clean the extracted article
cleaned = clean_text(article_text)


# Split into sentences
sentences = re.split(
    r"(?<=[.!?])\s+",
    cleaned
)

sentences = [
    sentence.strip()
    for sentence in sentences
    if len(sentence.strip()) >= 25
]


# Remove duplicate sentences
unique_sentences = []
seen = set()

for sentence in sentences:

    key = sentence.lower()

    if key not in seen:
        unique_sentences.append(sentence)
        seen.add(key)


TARGET_LINES = 250


# --------------------------------------------------
# First attempt: split sentences into ~15-word lines
# --------------------------------------------------

lines = []

for sentence in unique_sentences:

    words = sentence.split()

    for i in range(0, len(words), 15):

        chunk = " ".join(words[i:i + 15]).strip()

        if len(chunk) >= 25:
            lines.append(chunk)


# --------------------------------------------------
# If fewer than 200 lines, use ~10-word chunks
# --------------------------------------------------

if len(lines) < 200:

    lines = []

    for sentence in unique_sentences:

        words = sentence.split()

        for i in range(0, len(words), 10):

            chunk = " ".join(words[i:i + 10]).strip()

            if len(chunk) >= 20:
                lines.append(chunk)


# --------------------------------------------------
# If still fewer than 200 lines, use ~7-word chunks
# --------------------------------------------------

if len(lines) < 200:

    lines = []

    for sentence in unique_sentences:

        words = sentence.split()

        for i in range(0, len(words), 7):

            chunk = " ".join(words[i:i + 7]).strip()

            if len(chunk) >= 15:
                lines.append(chunk)


# Keep approximately 250 lines
lines = lines[:TARGET_LINES]


print("Generated dataset lines:", len(lines))


# Make sure we have enough text
if len(lines) < 200:

    print("\nThe extracted article is still too short.")
    print("Sentences extracted:", len(unique_sentences))
    print("Words extracted:", len(cleaned.split()))

    raise RuntimeError(
        "Not enough source text was extracted. "
        "Run the article extraction cell again."
    )


# --------------------------------------------------
# Save the dataset
# --------------------------------------------------

DATASET_FILE = "man_city_115_style_dataset.txt"

with open(
    DATASET_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(lines)
    )


print()
print("Dataset successfully created.")
print("File:", DATASET_FILE)
print("Number of lines:", len(lines))


print("\nSample lines:\n")

for i, line in enumerate(lines[:10], 1):

    print(
        "{}. {}".format(
            i,
            line
        )
    )

Generated dataset lines: 224

Dataset successfully created.
File: man_city_115_style_dataset.txt
Number of lines: 224

Sample lines:

1. An independent Commission has found Manchester City
2. FC guilty of all charges related to
3. serious breaches of the Premier League’s financial
4. rules over a nine-season period, and the
5. majority of charges in relation to its
6. failure to co-operate with the League’s investigation.
7. The independent Commission found that between Seasons
8. 2009/10 and 2017/18: - Manchester City arranged
9. ‘sham’ contracts (which misrepresented the true agreement
10. between the parties) with a number of


## 3. Inspect the dataset

The dataset should contain only language text. This is the cleaned custom style that will be used for fine-tuning.


In [9]:
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    dataset_lines = [line.strip() for line in f if line.strip()]

print("Number of training lines:", len(dataset_lines))
print("Average words per line:", round(sum(len(x.split()) for x in dataset_lines) / len(dataset_lines), 2))

for i, line in enumerate(dataset_lines[:10], 1):
    print(f"{i}: {line}")


Number of training lines: 224
Average words per line: 6.8
1: An independent Commission has found Manchester City
2: FC guilty of all charges related to
3: serious breaches of the Premier League’s financial
4: rules over a nine-season period, and the
5: majority of charges in relation to its
6: failure to co-operate with the League’s investigation.
7: The independent Commission found that between Seasons
8: 2009/10 and 2017/18: - Manchester City arranged
9: ‘sham’ contracts (which misrepresented the true agreement
10: between the parties) with a number of


## 4. Load GPT-2

GPT-2 is the same base model used in Assignment 2. The prompt from Assignment 2 was **`Artificial Intelligence is changing`**, so that exact prompt will be reused after fine-tuning.


In [10]:
import torch
from datasets import Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

MODEL_NAME = "gpt2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# GPT-2 has no separate padding token, so EOS is used for padding.
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id


Device: cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 5. Prepare the Hugging Face dataset

Each cleaned line is treated as a training example. The `DataCollatorForLanguageModeling` prepares labels for causal language-model training.


In [11]:
train_dataset = Dataset.from_dict({"text": dataset_lines})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(tokenized_dataset)


Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 224
})


## 6. Fine-tune GPT-2 with Hugging Face Trainer

The settings below are intentionally moderate so the notebook can be run in Google Colab. Training time depends on whether a GPU is available.


In [13]:
OUTPUT_DIR = "./gpt2_man_city_115"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=5e-5,

    weight_decay=0.01,

    logging_steps=10,

    save_strategy="no",

    report_to="none",

    fp16=torch.cuda.is_available()
)


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset,

    data_collator=data_collator
)


print("Starting GPT-2 fine-tuning...")

trainer.train()


trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)


print()
print("Fine-tuned model saved to:")
print(OUTPUT_DIR)

Starting GPT-2 fine-tuning...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.608887
20,4.245872
30,3.739360
40,3.362779
50,3.272467
60,3.197705
70,2.920282
80,3.049560


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved to:
./gpt2_man_city_115


## 7. Generate from the original GPT-2 and the fine-tuned GPT-2

This uses the **same Assignment 2 prompt** for both models, as required.


In [14]:
ASSIGNMENT_2_PROMPT = "Artificial Intelligence is changing"

BASE_MODEL_NAME = "gpt2"

base_tokenizer = GPT2Tokenizer.from_pretrained(BASE_MODEL_NAME)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = GPT2LMHeadModel.from_pretrained(BASE_MODEL_NAME).to(device)
base_model.eval()

finetuned_tokenizer = GPT2Tokenizer.from_pretrained(OUTPUT_DIR)
finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
finetuned_model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).to(device)
finetuned_model.eval()


def generate_text(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


base_output = generate_text(
    base_model,
    base_tokenizer,
    ASSIGNMENT_2_PROMPT
)

finetuned_output = generate_text(
    finetuned_model,
    finetuned_tokenizer,
    ASSIGNMENT_2_PROMPT
)

print("=" * 80)
print("BASE GPT-2 OUTPUT")
print("=" * 80)
print(base_output)

print("\n" + "=" * 80)
print("FINE-TUNED GPT-2 OUTPUT")
print("=" * 80)
print(finetuned_output)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BASE GPT-2 OUTPUT
Artificial Intelligence is changing the world, and it's not just about solving real problems. It's also trying to create a new form of human interaction with robots — one that will be fundamentally different from what we've been used in our everyday lives for far too long.
: The way AI works has already changed many things when compared to previous generations. For example, recent developments have allowed us to interact more realistically through software

FINE-TUNED GPT-2 OUTPUT
Artificial Intelligence is changing the way we see and interact with our customers. The company believes that it has a right to publish accurate, current information about its business practices in relation as well of all relevant documents filed under Section 12(b)(3) (which were amended from time until date). This process cannot be described by simply stating "in these circumstances", or through an independent Commission’s decision ‘over such


## 8. Save the outputs for submission


In [15]:
with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(base_output)

with open("finetuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(finetuned_output)

print("Saved:")
print("- base_model_output.txt")
print("- finetuned_model_output.txt")
print("- " + DATASET_FILE)


Saved:
- base_model_output.txt
- finetuned_model_output.txt
- man_city_115_style_dataset.txt


## 9. Before-and-after comparison

### Expected style differences to look for

The fine-tuned model should be evaluated against the actual output produced in the previous cell. Do not claim that a difference occurred unless it is visible in the generated text.

| Aspect | Base GPT-2 | Fine-tuned GPT-2 |
|---|---|---|
| Vocabulary | General-purpose wording | Look for football, regulatory and financial terminology |
| Tone | General / conversational continuation | Look for formal, institutional and disciplinary language |
| Sentence structure | General GPT-2 continuation patterns | Look for longer formal sentences and report-style phrasing |
| Topic tendency | Broad continuation around AI | Look for Premier League, clubs, charges, rules, investigations or sanctions |

### Two clear differences to report
1. **Vocabulary/style:** The fine-tuned model may use terminology associated with football regulation, financial rules, investigations and disciplinary proceedings more often than the base model.
2. **Tone/structure:** The fine-tuned model may produce more formal, institutional and report-like sentences, reflecting the source material.

Replace the statements above with observations based on your actual outputs if the generated text shows something different.


## 10. Final conclusion

Fine-tuning adapts the pretrained GPT-2 model toward the vocabulary, tone and sentence patterns present in the custom dataset. In this experiment, the custom dataset represents the formal style of the Premier League's statement about Manchester City FC. The comparison between the original and fine-tuned outputs demonstrates how a relatively small, consistent corpus can shift a general language model toward a specialised writing style.

